# CNN PyTorch

Nguyễn Ngọc Hoàng Nam - B23DCCN585 | Assignment 04

**Bản mở rộng:** notebook này giữ nhóm 18 lượt seed 42 để giải thích thuật toán. Notebook **06** tổng hợp đầy đủ 54 lượt nhiều seed và 18 ablation; notebook **07** thực thi ví dụ số học dùng trong báo cáo 78 trang.

In [1]:
from pathlib import Path
import os, sys, json, subprocess
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT / 'src').is_dir(), 'Hãy mở notebook từ thư mục repository.'
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from IPython.display import display, Markdown, Image
from src.data import DATASETS, load_data, data_root
print('Python:', sys.version.split()[0])
print('Dữ liệu:', data_root())

Python: 3.10.20
Dữ liệu: E:\PTHTTM\ASG_04_data


## 1. Cách triển khai

Mô hình dùng torch.nn và autograd. BatchNorm được viết bằng phép toán tensor để khớp phương sai tổng thể của bản NumPy. Huấn luyện dùng loss.backward() và torch.optim.Adam.

Mã trong các ô dưới lấy trực tiếp từ module trong `src/`. Vòng lặp huấn luyện lưu checkpoint theo validation loss và chỉ đánh giá test sau khi khôi phục checkpoint tốt nhất.

In [2]:
BACKEND = 'pytorch'
RETRAIN = False  # Đổi thành True để huấn luyện lại 6 cấu hình; sẽ ghi đè kết quả tương ứng.

## 2. Mã mô hình

In [3]:
"""PyTorch equivalent of the NumPy CNN (including population-variance BatchNorm)."""
import torch
from torch import nn

class PopulationBatchNorm(nn.Module):
    """Match NumPy/TF running variance; PyTorch's standard layer uses an unbiased estimate."""
    def __init__(self,c):
        super().__init__();self.weight=nn.Parameter(torch.ones(c));self.bias=nn.Parameter(torch.zeros(c))
        self.register_buffer('running_mean',torch.zeros(c));self.register_buffer('running_var',torch.ones(c))
    def forward(self,x):
        if self.training:
            mean=x.mean((0,2,3));var=x.var((0,2,3),unbiased=False)
            with torch.no_grad():
                self.running_mean.mul_(0.9).add_(mean.detach(),alpha=0.1)
                self.running_var.mul_(0.9).add_(var.detach(),alpha=0.1)
        else:mean,var=self.running_mean,self.running_var
        return (x-mean[None,:,None,None])*torch.rsqrt(var[None,:,None,None]+1e-5)*self.weight[None,:,None,None]+self.bias[None,:,None,None]

class Residual(nn.Module):
    def __init__(self,c):
        super().__init__();self.conv=nn.Conv2d(c,c,3,padding=1);self.bn=PopulationBatchNorm(c);self.use_skip=True
    def forward(self,x):
        branch=self.bn(self.conv(x))
        return torch.relu(branch+x if self.use_skip else branch)

class PopulationBatchNorm1D(PopulationBatchNorm):
    def forward(self,x):return super().forward(x[:,:,None,None])[:,:,0,0]

class TorchCNN(nn.Module):
    def __init__(self,channels,size,classes,variant='baseline'):
        super().__init__();improved=variant=='improved'
        layers=[nn.Conv2d(channels,8,3,padding=1)]
        if improved:layers.append(PopulationBatchNorm(8))
        layers += [nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(8,16,3,padding=1)]
        if improved:layers.append(PopulationBatchNorm(16))
        layers.append(nn.ReLU())
        if improved:layers.append(Residual(16))
        layers += [nn.MaxPool2d(2),nn.Flatten(),nn.Linear(16*(size//4)**2,64)]
        if improved:layers.append(PopulationBatchNorm1D(64))
        layers.append(nn.ReLU())
        if improved:layers.append(nn.Dropout(0.25))
        layers.append(nn.Linear(64,classes));self.layers=nn.Sequential(*layers)
    def forward(self,x):return self.layers(x)
    def ablate(self,component):
        """Apply after shared initialization; preserve convolution and Dense weights."""
        if component=='no_bn':
            for i,layer in enumerate(self.layers):
                if isinstance(layer,PopulationBatchNorm):self.layers[i]=nn.Identity()
                elif isinstance(layer,Residual):layer.bn=nn.Identity()
        elif component=='no_skip':
            for layer in self.layers:
                if isinstance(layer,Residual):layer.use_skip=False
        elif component=='no_dropout':
            for layer in self.layers:
                if isinstance(layer,nn.Dropout):layer.p=0.0
        elif component is not None:raise ValueError(component)
        return self
    def load_numpy(self,state):
        with torch.no_grad():
            for i,layer in enumerate(self.layers):
                prefix=f'{i}.'
                if isinstance(layer,(nn.Conv2d,nn.Linear)):
                    w=state[prefix+'weight'];w=w.T if isinstance(layer,nn.Linear) else w
                    layer.weight.copy_(torch.from_numpy(w.copy()));layer.bias.copy_(torch.from_numpy(state[prefix+'bias']))
                elif isinstance(layer,PopulationBatchNorm):
                    self._bn(layer,state,prefix)
                elif isinstance(layer,Residual):
                    layer.conv.weight.copy_(torch.from_numpy(state[prefix+'conv_weight']))
                    layer.conv.bias.copy_(torch.from_numpy(state[prefix+'conv_bias']))
                    self._bn(layer.bn,state,prefix+'bn_')
    @staticmethod
    def _bn(layer,state,p):
        for a,b in [('weight','gamma'),('bias','beta'),('running_mean','running_mean'),('running_var','running_var')]:
            getattr(layer,a).copy_(torch.from_numpy(state[p+b]))


## 3. Vòng lặp huấn luyện và đánh giá

Chương trình bên dưới chứa đầy đủ bước lấy batch, forward, loss, gradient, cập nhật, validation, checkpoint và đánh giá test. Các lượt chạy thực tế gọi cùng module trong một tiến trình riêng để cô lập framework.

In [4]:
"""Reproducible full-dataset training. Run: python -m src.train --help."""
import os
for key in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS']:
    os.environ.setdefault(key,'1')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL','2')
import json,csv,time,platform,argparse,sys,copy
from pathlib import Path
import numpy as np
from src.data import ROOT,DATASETS,load_data,batches
from src.numpy_cnn import CNN,cross_entropy,Adam

def classification_metrics(y,logits,k):
    pred=logits.argmax(axis=1);cm=np.bincount(y*k+pred,minlength=k*k).reshape(k,k)
    tp=np.diag(cm).astype(float)
    precision=np.divide(tp,cm.sum(0),out=np.zeros(k),where=cm.sum(0)>0)
    recall=np.divide(tp,cm.sum(1),out=np.zeros(k),where=cm.sum(1)>0)
    f1=np.divide(2*precision*recall,precision+recall,out=np.zeros(k),where=(precision+recall)>0)
    result={'accuracy':float((pred==y).mean()),'macro_precision':float(precision.mean()),'macro_recall':float(recall.mean()),'macro_f1':float(f1.mean()),'test_loss':cross_entropy(logits,y)[0]}
    if k>=5:result['top5_accuracy']=float(np.any(np.argsort(logits,axis=1)[:,-5:]==y[:,None],axis=1).mean())
    return result,cm

def experiment_dir(backend,dataset,variant,seed=42,ablation=None):
    name=f'{dataset}_{backend}_{variant}'
    if ablation:return ROOT/'results'/'ablation'/f'seed_{seed}'/(name+'_'+ablation)
    if seed!=42:return ROOT/'results'/'multiseed'/f'seed_{seed}'/name
    return ROOT/'results'/name

def _train(backend,dataset,variant,epochs=None,batch_size=128,seed=42,force=False,ablation=None):
    if ablation and (backend!='pytorch' or variant!='improved'):
        raise ValueError('Ablation requires the PyTorch improved architecture.')
    cfg=DATASETS[dataset];epochs=epochs or cfg['epochs']
    out=experiment_dir(backend,dataset,variant,seed,ablation);out.mkdir(parents=True,exist_ok=True)
    if (out/'metrics.json').exists() and not force:
        saved=json.loads((out/'config.json').read_text())
        for key,value in dict(seed=seed,epochs=epochs,batch_size=batch_size,ablation=ablation).items():
            if saved.get(key)!=value:raise ValueError(f'Existing {out}: {key} differs; use --force or another seed.')
        return json.loads((out/'metrics.json').read_text())
    data=load_data(dataset);x,y=data['x'],data['y'];ti,vi=data['train_ids'],data['val_ids']
    spec=dict(channels=cfg['channels'],size=cfg['size'],classes=cfg['classes'],variant=variant,seed=seed)
    initial=CNN(**spec);state=initial.state();params=initial.parameter_count()
    np.random.seed(seed)
    if backend=='numpy':
        model=initial;optimizer=Adam();device='CPU';version=np.__version__
        def predict(b):return model.forward(b,False)
        def step(b,t):
            logits=model.forward(b,True);loss,g=cross_entropy(logits,t);model.backward(g);optimizer.step(model.params())
            return loss,int((logits.argmax(1)==t).sum())
        def save():np.savez_compressed(out/'weights.npz',**model.state())
        def restore():
            with np.load(out/'weights.npz') as d:model.load_state(dict(d))
    elif backend=='pytorch':
        import torch
        from src.torch_cnn import TorchCNN
        torch.manual_seed(seed);torch.set_num_threads(4)
        torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True
        torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False
        device='cuda' if torch.cuda.is_available() else 'cpu';version=torch.__version__
        model=TorchCNN(cfg['channels'],cfg['size'],cfg['classes'],variant);model.load_numpy(state);model.to(device)
        assert sum(p.numel() for p in model.parameters())==params
        model.ablate(ablation);params=sum(p.numel() for p in model.parameters())
        optimizer=torch.optim.Adam(model.parameters(),lr=1e-3,eps=1e-8)
        def predict(b):
            model.eval()
            with torch.no_grad():return model(torch.from_numpy(np.ascontiguousarray(b)).to(device)).cpu().numpy()
        def step(b,t):
            model.train();optimizer.zero_grad(set_to_none=True)
            logits=model(torch.from_numpy(np.ascontiguousarray(b)).to(device));target=torch.from_numpy(t).to(device)
            loss=torch.nn.functional.cross_entropy(logits,target);loss.backward();optimizer.step()
            return loss.item(),int((logits.argmax(1)==target).sum().item())
        def save():torch.save(model.state_dict(),out/'weights.pt')
        def restore():model.load_state_dict(torch.load(out/'weights.pt',map_location=device,weights_only=True))
    elif backend=='tensorflow':
        cuda=os.environ.get('CNN_CUDA_DIR','E:/PTHTTM/ASG_04_runtime/cuda/Library/bin')
        if os.name=='nt' and Path(cuda).is_dir():
            os.environ['PATH']=cuda+os.pathsep+os.environ['PATH'];dll=os.add_dll_directory(cuda)
        import tensorflow as tf
        from src.tf_cnn import build_model,load_numpy
        tf.keras.utils.set_random_seed(seed)
        tf.config.threading.set_intra_op_parallelism_threads(4);tf.config.threading.set_inter_op_parallelism_threads(2)
        tf.config.experimental.enable_tensor_float_32_execution(False)
        gpus=tf.config.list_physical_devices('GPU')
        for gpu in gpus:tf.config.experimental.set_memory_growth(gpu,True)
        device='GPU' if gpus else 'CPU';version=tf.__version__
        model=build_model(cfg['channels'],cfg['size'],cfg['classes'],variant);load_numpy(model,state,variant)
        assert sum(int(np.prod(v.shape)) for v in model.trainable_weights)==params
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3,epsilon=1e-8)
        @tf.function(reduce_retracing=True)
        def train_step(b,t):
            with tf.GradientTape() as tape:
                logits=model(b,training=True)
                loss=tf.reduce_mean(tf.nn.sparse_softmax_cross_entropy_with_logits(labels=t,logits=logits))
            optimizer.apply_gradients(zip(tape.gradient(loss,model.trainable_weights),model.trainable_weights))
            return loss,tf.reduce_sum(tf.cast(tf.argmax(logits,axis=1)==t,tf.int32))
        @tf.function(reduce_retracing=True)
        def infer(b):return model(b,training=False)
        def predict(b):return infer(np.ascontiguousarray(b.transpose(0,2,3,1))).numpy()
        def step(b,t):
            loss,correct=train_step(np.ascontiguousarray(b.transpose(0,2,3,1)),t)
            return float(loss.numpy()),int(correct.numpy())
        def save():model.save_weights(str(out/'weights.h5'))
        def restore():model.load_weights(str(out/'weights.h5'))
    else:raise ValueError(backend)
    history=[];best=float('inf');best_epoch=0;train_seconds=0;val_seconds=0
    print(f'START {dataset} {backend} {variant} seed={seed} ablation={ablation}: train={len(ti)} val={len(vi)} epochs={epochs} device={device} params={params}',flush=True)
    config=dict(dataset=dataset,backend=backend,variant=variant,epochs=epochs,batch_size=batch_size,seed=seed,learning_rate=0.001,optimizer='Adam',adam_beta1=0.9,adam_beta2=0.999,adam_epsilon=1e-8,normalization='uint8 / 255',train_samples=len(ti),validation_samples=len(vi),test_samples=len(data['y_test']),device=device,framework_version=version,python=sys.version,platform=platform.platform(),parameter_count=params,selection='minimum validation cross-entropy',augmentation=False,training_scope='all predefined training samples, no subsampling')
    from threadpoolctl import threadpool_info
    config['ablation']=ablation
    config['split_seed']=42
    config['cpu_thread_pools']=threadpool_info()
    (out/'config.json').write_text(json.dumps(config,indent=2),encoding='utf-8')
    for epoch in range(1,epochs+1):
        ts=time.perf_counter();ls=0;correct=0
        for b,t in batches(x,y,ti,batch_size,seed+epoch):
            loss,c=step(b,t);ls+=loss*len(t);correct+=c
        elapsed=time.perf_counter()-ts;train_seconds+=elapsed;ts=time.perf_counter()
        logits=np.concatenate([predict(b) for b,t in batches(x,y,vi,batch_size)])
        vl=cross_entropy(logits,y[vi])[0];va=float((logits.argmax(1)==y[vi]).mean());ve=time.perf_counter()-ts;val_seconds+=ve
        if not np.isfinite(ls+vl):raise FloatingPointError('Non-finite loss. Stop rather than record invalid results.')
        if vl<best:best=vl;best_epoch=epoch;save()
        row=dict(epoch=epoch,train_loss=ls/len(ti),train_accuracy=correct/len(ti),val_loss=vl,val_accuracy=va,train_seconds=elapsed,validation_seconds=ve)
        history.append(row)
        with (out/'history.csv').open('w',newline='',encoding='utf-8') as f:
            writer=csv.DictWriter(f,fieldnames=list(row));writer.writeheader();writer.writerows(history)
        print(f'{dataset}/{backend}/{variant} epoch {epoch}/{epochs}: loss {row["train_loss"]:.4f} val_loss {vl:.4f} val_acc {va:.4f} train_s {elapsed:.1f}',flush=True)
    restore();xt,yt=data['x_test'],data['y_test'];ts=time.perf_counter()
    logits=np.concatenate([predict(b) for b,t in batches(xt,yt,np.arange(len(yt)),batch_size)])
    infer_seconds=time.perf_counter()-ts
    metrics,cm=classification_metrics(yt,logits,cfg['classes'])
    metrics.update({k:config[k] for k in ['dataset','backend','variant','parameter_count','train_samples','validation_samples','test_samples','device']})
    metrics.update(seed=seed,ablation=ablation,epochs=epochs,best_epoch=best_epoch,best_val_loss=best,train_seconds=train_seconds,validation_seconds=val_seconds,test_seconds=infer_seconds)
    shifted=logits-logits.max(1,keepdims=True);probs=np.exp(shifted);probs/=probs.sum(1,keepdims=True)
    with (out/'predictions.csv').open('w',newline='',encoding='utf-8') as f:
        writer=csv.writer(f);writer.writerow(['test_id','true_label','predicted_label','confidence'])
        writer.writerows(zip(range(len(yt)),yt.tolist(),logits.argmax(1).tolist(),probs.max(1).tolist()))
    np.savez_compressed(out/'test_outputs.npz',logits=logits,labels=yt)
    np.savetxt(out/'confusion_matrix.csv',cm,delimiter=',',fmt='%d')
    (out/'metrics.json').write_text(json.dumps(metrics,indent=2),encoding='utf-8')
    print('COMPLETE '+json.dumps(metrics),flush=True)
    return metrics

def train(backend,dataset,variant,epochs=None,batch_size=128,seed=42,force=False,ablation=None):
    # A notebook and the CLI may request the same configuration concurrently.
    # Serialize that configuration; after waiting, reuse its complete results.
    from filelock import FileLock
    folder=experiment_dir(backend,dataset,variant,seed,ablation)
    folder.mkdir(parents=True,exist_ok=True)
    with FileLock(str(folder/'.train.lock'),timeout=7200):
        return _train(backend,dataset,variant,epochs,batch_size,seed,force,ablation)



In [5]:
def execute_dataset(name):
    rows=[]
    for variant in ['baseline','improved']:
        folder=ROOT/'results'/f'{name}_{BACKEND}_{variant}'
        if RETRAIN or not (folder/'metrics.json').exists():
            cmd=[sys.executable,'-m','src.train','--backend',BACKEND,'--dataset',name,'--variant',variant]
            if RETRAIN:cmd.append('--force')
            subprocess.run(cmd,cwd=ROOT,check=True)
        else:
            print('Đọc kết quả đã huấn luyện:',folder.name)
        rows.append(json.loads((folder/'metrics.json').read_text()))
        display(pd.read_csv(folder/'history.csv'))
    display(pd.DataFrame(rows)[['dataset','variant','accuracy','macro_f1','test_loss','top5_accuracy','best_epoch','train_seconds']])
    return rows

## 4. MNIST

In [6]:
mnist_results=execute_dataset('mnist')

Đọc kết quả đã huấn luyện: mnist_pytorch_baseline


,epoch,train_loss,train_accuracy,val_loss,val_accuracy,train_seconds,validation_seconds
0,1,0.266759,0.922853,0.106884,0.966161,8.898281,0.048949
1,2,0.079266,0.975260,0.077414,0.976329,1.745210,0.051253
2,3,0.056656,0.983185,0.069719,0.977830,1.736148,0.049831
3,4,0.044760,0.986519,0.064084,0.979497,1.687888,0.061690
4,5,0.038237,0.988056,0.052562,0.983664,1.812054,0.058105


Đọc kết quả đã huấn luyện: mnist_pytorch_improved


,epoch,train_loss,train_accuracy,val_loss,val_accuracy,train_seconds,validation_seconds
0,1,0.260149,0.931483,0.081647,0.977163,8.810050,0.118626
1,2,0.080482,0.978056,0.055188,0.983831,4.500668,0.120091
2,3,0.055336,0.983741,0.052063,0.983831,9.502287,0.308534
3,4,0.044407,0.986889,0.043785,0.986498,10.997059,0.288092
4,5,0.036630,0.988759,0.042016,0.987665,10.030871,0.229548


,dataset,variant,accuracy,macro_f1,test_loss,top5_accuracy,best_epoch,train_seconds
0,mnist,baseline,0.9863,0.986174,0.040365,0.9997,5,15.879581
1,mnist,improved,0.9903,0.990208,0.030867,0.9999,5,43.840935


## 5. CIFAR-10

In [7]:
cifar10_results=execute_dataset('cifar10')

Đọc kết quả đã huấn luyện: cifar10_pytorch_baseline


,epoch,train_loss,train_accuracy,val_loss,val_accuracy,train_seconds,validation_seconds
0,1,1.659788,0.404844,1.464538,0.4728,9.542268,0.242694
1,2,1.375754,0.513511,1.328884,0.5388,3.634654,0.173734
2,3,1.270301,0.551956,1.269270,0.5540,4.026936,0.241082
3,4,1.199767,0.579644,1.197106,0.5780,4.074659,0.218354
4,5,1.151516,0.595667,1.166112,0.5880,3.761841,0.178284
5,6,1.098653,0.613578,1.140956,0.6020,3.370295,0.161847
6,7,1.064338,0.627667,1.109737,0.6100,3.402118,0.325398
7,8,1.029994,0.640000,1.091177,0.6150,3.776218,0.217259
8,9,1.004120,0.650578,1.084625,0.6136,4.785900,0.364743
9,10,0.974818,0.658822,1.076832,0.6214,4.748843,0.218873


Đọc kết quả đã huấn luyện: cifar10_pytorch_improved


,epoch,train_loss,train_accuracy,val_loss,val_accuracy,train_seconds,validation_seconds
0,1,1.580950,0.436800,1.322766,0.5248,14.340202,0.312232
1,2,1.252190,0.555333,1.136371,0.6022,9.053327,0.275167
2,3,1.125657,0.601533,1.108567,0.6052,8.941536,0.314472
3,4,1.045109,0.629511,1.097265,0.6074,8.745666,0.274070
4,5,0.991508,0.650756,0.986785,0.6524,8.688805,0.346541
5,6,0.950482,0.663467,1.049074,0.6314,8.788847,0.282748
6,7,0.910914,0.677467,0.984221,0.6530,9.134471,0.360914
7,8,0.880030,0.689200,0.994959,0.6554,8.789555,0.287089
8,9,0.850647,0.697311,0.973966,0.6640,9.025433,0.381308
9,10,0.828380,0.705711,1.006161,0.6494,8.782400,0.323648


,dataset,variant,accuracy,macro_f1,test_loss,top5_accuracy,best_epoch,train_seconds
0,cifar10,baseline,0.6303,0.626638,1.061751,0.9624,10,45.123732
1,cifar10,improved,0.6526,0.653112,0.999070,0.9670,9,94.290242


## 6. CIFAR-100

In [8]:
cifar100_results=execute_dataset('cifar100')

Đọc kết quả đã huấn luyện: cifar100_pytorch_baseline


,epoch,train_loss,train_accuracy,val_loss,val_accuracy,train_seconds,validation_seconds
0,1,4.174596,0.071733,3.757410,0.1304,9.509691,0.260279
1,2,3.565266,0.165733,3.433137,0.1790,4.243458,0.232391
2,3,3.314254,0.209844,3.300737,0.2084,4.261126,0.219332
3,4,3.167492,0.236600,3.203821,0.2252,4.327223,0.232807
4,5,3.054295,0.257644,3.132705,0.2382,4.349760,0.226197
5,6,2.977351,0.271044,3.095660,0.2490,4.479771,0.234261
6,7,2.900455,0.288867,3.054120,0.2528,4.205435,0.202973
7,8,2.845394,0.296422,3.070622,0.2452,4.128426,0.261171
8,9,2.794560,0.307711,2.990022,0.2582,4.402287,0.218331
9,10,2.751330,0.315022,2.945698,0.2736,4.565240,0.230414


Đọc kết quả đã huấn luyện: cifar100_pytorch_improved


,epoch,train_loss,train_accuracy,val_loss,val_accuracy,train_seconds,validation_seconds
0,1,4.130862,0.088867,3.638984,0.1582,16.626238,0.339058
1,2,3.542523,0.165911,3.289893,0.2204,9.522166,0.384904
2,3,3.263309,0.213889,3.087081,0.2554,10.327140,0.386328
3,4,3.087130,0.244022,2.994311,0.2624,10.133899,0.315838
4,5,2.964977,0.267178,2.936092,0.2712,8.903090,0.329114
5,6,2.872596,0.286467,2.854499,0.2874,8.929971,0.276828
6,7,2.790423,0.300756,2.794572,0.2966,8.976731,0.311173
7,8,2.722860,0.312956,2.824246,0.2946,8.252587,0.273801
8,9,2.671166,0.325067,2.727797,0.3118,7.796820,0.269285
9,10,2.611976,0.334600,2.783217,0.3014,8.036775,0.263748


,dataset,variant,accuracy,macro_f1,test_loss,top5_accuracy,best_epoch,train_seconds
0,cifar100,baseline,0.2884,0.277303,2.885165,0.5888,12,56.890921
1,cifar100,improved,0.3215,0.309822,2.697512,0.6235,11,113.416461


## 7. Kiểm tra triển khai

In [9]:
verification=ROOT/'results'/f'verification_{BACKEND}.json'
if verification.exists():
    display(pd.DataFrame(json.loads(verification.read_text())))
else:
    print(subprocess.run([sys.executable,'tools/run_test_suite.py'],cwd=ROOT,capture_output=True,text=True,check=True).stderr)

,backend,dataset,variant,passed,max_absolute_errors
0,pytorch,mnist,baseline,True,"{'train_logits': 1.1920928955078125e-06, 'inpu..."
1,pytorch,mnist,improved,True,"{'train_logits': 5.0067901611328125e-06, 'inpu..."
2,pytorch,cifar10,baseline,True,"{'train_logits': 1.4901161193847656e-06, 'inpu..."
3,pytorch,cifar10,improved,True,"{'train_logits': 4.112720489501953e-06, 'input..."
4,pytorch,cifar100,baseline,True,"{'train_logits': 3.5762786865234375e-06, 'inpu..."
5,pytorch,cifar100,improved,True,"{'train_logits': 4.887580871582031e-06, 'input..."


## Diễn giải

So sánh baseline với improved trong cùng dataset/backend. Accuracy không phản ánh toàn bộ chất lượng ở CIFAR-100: cần xem macro-F1, top-5 và các lớp hay nhầm. Train metrics được tích lũy trong lúc cập nhật trọng số, có dropout ở improved; validation chạy ở chế độ eval, nên không thể diễn giải mọi chênh lệch train-validation là overfitting.

Các chỉ số là một lượt chạy seed 42. Thời gian gồm bước huấn luyện thực tế nhưng không phải benchmark phần cứng độc lập. Notebook 05 đưa ra so sánh chung dựa trên 18 kết quả.